<a href="https://colab.research.google.com/github/ene23033/repository/blob/master/10yrs_values_averaged%2BPOA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd



In [ ]:
df1 = pd.read_csv('/content/Maharashtra_parameters_solcast_2014-2023.csv')

df1= df1.drop(columns={'period_end','period'})
df1.head()

,dhi,dni,ghi,wind_direction_100m,wind_speed_100m
0,0,0,0,10,6.1
1,0,0,0,14,5.5
2,0,0,0,18,4.9
3,0,0,0,19,4.7
4,0,0,0,25,4.4


In [ ]:
start_date = pd.to_datetime('2014-01-01 00:00')
end_date = pd.to_datetime('2023-12-31 23:00')
date_range = pd.date_range(start=start_date, end=end_date, freq='h')
df2 = pd.DataFrame({'timestamp': date_range})
df2 = pd.to_datetime(df2['timestamp'],format= '%d-%m-%Y %H:%M')
df2.tail()

,timestamp
87643,2023-12-31 19:00:00
87644,2023-12-31 20:00:00
87645,2023-12-31 21:00:00
87646,2023-12-31 22:00:00
87647,2023-12-31 23:00:00


In [ ]:
frames = [df2,df1]
df = pd.concat(frames) #On doing this, it added the df2 to certain row, then added df2 after, so I cut the df2 rows in excel and pasted adjacently to df1 rows.
df.head()
df.to_csv('mergeddf1df2.csv',index=False)

In [ ]:
df = pd.read_csv('/content/mergeddf1df2.csv',parse_dates=['timestamp'])

In [ ]:
df.head()

,timestamp,dhi,dni,ghi,wind_direction_100m,wind_speed_100m
0,01-01-2014 00:00,0,0,0,10,6.1
1,01-01-2014 01:00,0,0,0,14,5.5
2,01-01-2014 02:00,0,0,0,18,4.9
3,01-01-2014 03:00,0,0,0,19,4.7
4,01-01-2014 04:00,0,0,0,25,4.4


In [ ]:
# Ensure timestamp is in correct datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M')

# Extract month, day, and hour to group by
df['month_day_hour'] = df['timestamp'].dt.strftime('%m-%d %H:%M')

# Compute the average for each unique month-day-hour combination
average_df = df.groupby('month_day_hour')[['dhi', 'dni', 'ghi', 'wind_direction_100m', 'wind_speed_100m']].mean().reset_index()

# Convert 'month_day_hour' back to a standard datetime format (using a reference year, e.g., 2000)
average_df['timestamp'] = pd.to_datetime('2000-' + average_df['month_day_hour'], format='%Y-%m-%d %H:%M')

# Drop the helper column
average_df = average_df.drop(columns=['month_day_hour'])

# Save the result to a new CSV file
average_df.to_csv('averaged_2014-2030_sol_wind_data.csv', index=False)


In [25]:
df1= pd.read_csv('/content/averaged_2014-2023_sol_wind_data.csv')

df1.index = pd.to_datetime(df1['timestamp'], format='%d-%m-%Y %H:%M')

df1.head()

,timestamp,dhi,dni,ghi,wind_direction_100m,wind_speed_100m
timestamp,,,,,,
2030-01-01 00:00:00,01-01-2030 00:00,0.0,0.0,0.0,59.0,3.68
2030-01-01 01:00:00,01-01-2030 01:00,0.0,0.0,0.0,62.2,3.34
2030-01-01 02:00:00,01-01-2030 02:00,0.0,0.0,0.0,65.3,3.15
2030-01-01 03:00:00,01-01-2030 03:00,0.0,0.0,0.0,101.8,3.24
2030-01-01 04:00:00,01-01-2030 04:00,0.0,0.0,0.0,105.3,3.30


In [26]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2030-01-01 00:00:00 to 2030-12-31 23:00:00
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   timestamp            8760 non-null   object 
 1   dhi                  8760 non-null   float64
 2   dni                  8760 non-null   float64
 3   ghi                  8760 non-null   float64
 4   wind_direction_100m  8760 non-null   float64
 5   wind_speed_100m      8760 non-null   float64
dtypes: float64(5), object(1)
memory usage: 479.1+ KB


In [32]:
!pip install pvlib
!pip install TimezoneFinder

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pvlib.location import Location
from timezonefinder import TimezoneFinder
import pvlib

In [29]:
tilt = 19
azimuth = 180

location_latitude = 19.1331
location_longitude = 72.9151

tf = TimezoneFinder()
tz = tf.certain_timezone_at(lat=location_latitude, lng=location_longitude)
print(tz)

df1.index = df1.index.tz_localize(tz)

loc = Location(location_latitude, location_longitude, tz)

sun = loc.get_solarposition(df1.index)


Asia/Kolkata


In [30]:
s = pd.merge(df1, sun, left_on=df1.index, right_on=sun.index)
s.head()

,key_0,timestamp,dhi,dni,ghi,wind_direction_100m,wind_speed_100m,apparent_zenith,zenith,apparent_elevation,elevation,azimuth,equation_of_time
0,2030-01-01 00:00:00+05:30,01-01-2030 00:00,0.0,0.0,0.0,59.0,3.68,169.553734,169.553734,-79.553734,-79.553734,246.289376,-3.227737
1,2030-01-01 01:00:00+05:30,01-01-2030 01:00,0.0,0.0,0.0,62.2,3.34,174.204426,174.204426,-84.204426,-84.204426,133.006240,-3.247530
2,2030-01-01 02:00:00+05:30,01-01-2030 02:00,0.0,0.0,0.0,65.3,3.15,161.321715,161.321715,-71.321715,-71.321715,105.450770,-3.267314
3,2030-01-01 03:00:00+05:30,01-01-2030 03:00,0.0,0.0,0.0,101.8,3.24,147.563781,147.563781,-57.563781,-57.563781,103.050621,-3.287091
4,2030-01-01 04:00:00+05:30,01-01-2030 04:00,0.0,0.0,0.0,105.3,3.30,133.780166,133.780166,-43.780166,-43.780166,103.931391,-3.306858


In [36]:
s['beam'] = pvlib.irradiance.beam_component(tilt, azimuth, s['zenith'], s['azimuth'], s['dni'])
s['sky'] = pvlib.irradiance.isotropic(tilt, s['dhi'])
s['NASA_poa'] = s['beam']+s['sky']

In [37]:
s.to_csv('poa+wind_averaged_2014-2023.csv', index=False)